In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.manifold import TSNE
import numpy as np
import os
import glasbey

pio.renderers.default = 'browser'

In [2]:
# CHANGE HERE: Path to your CSV file
csv_file = r'C:\Users\gangliagurdian\Desktop\Mias Folder\Poster Files\3D_Data\combined_matrix_final_total136_lc.csv'
df = pd.read_csv(csv_file)

In [3]:
# Compute t-SNE embedding (if not precomputed)
features = df.iloc[:, 0:30].values.astype(float)
tsne = TSNE(n_components=2, perplexity=30, learning_rate=100, random_state=42)
Y = tsne.fit_transform(features)
df['TSNE-1'] = Y[:, 0]
df['TSNE-2'] = Y[:, 1]

# Get all clusters present across the ENTIRE dataset
all_clusters = sorted(df['ClusterNumber'].unique())
cluster_labels_str = [str(c) for c in all_clusters]

In [4]:
# CHANGE HERE: Customize palette size to the number of clusters
glasbey_palette = glasbey.create_palette(palette_size=75)
color_num = len(glasbey_palette)
cluster_to_color = {
    str(c): glasbey_palette[i % color_num]
    for i, c in enumerate(all_clusters)
}

In [5]:
# 1. Master t-SNE plot (all data, colored consistently)
fig_all = px.scatter(
    df,
    x='TSNE-1', y='TSNE-2',
    color=df['ClusterNumber'].astype(str),
    color_discrete_map=cluster_to_color,
    hover_name=df['ClusterNumber'].astype(str),
    labels={'color': 'Cluster'},
    title='t-SNE of All Data'
)
fig_all.update_traces(marker=dict(size=7, line=dict(width=0.5, color='DarkSlateGrey')))
fig_all.update_layout(
    legend_title='Cluster',
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)

x_range = fig_all.layout.xaxis.range # Capture axis ranges from master plot
y_range = fig_all.layout.yaxis.range

# CHANGE HERE: Save path for the master plot
save_path = r"C:\Users\gangliagurdian\Desktop\Mias Folder\Poster Files\3D_Visualizations\Control\tSNE\all_data"
pio.write_image(fig_all, save_path + ".png", width=1000, height=1000)
pio.write_image(fig_all, save_path + ".svg", width=1000, height=1000)

fig_all.show() 

In [ ]:
# 2-4. Per-experiment type t-SNE plots, always using the full color map
# CHANGE HERE: Define exp_label to update titles of your plots
for exp_type, exp_label in zip(['week1', 'week3', 'week6'], ['Control Week 1', 'Control Week 3', 'Control Week 6']):
    df_exp = df[df['ExperimentType'] == exp_type].copy()
    if df_exp.empty:
        continue
    fig = px.scatter(
        df_exp,
        x='TSNE-1', y='TSNE-2',
        color=df_exp['ClusterNumber'].astype(str),
        color_discrete_map=cluster_to_color,
        category_orders={'ClusterNumber': cluster_labels_str},
        hover_name=df_exp['ClusterNumber'].astype(str),
        labels={'color': 'Cluster'},
        title=f't-SNE: {exp_label}'
    )
    fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color='DarkSlateGrey')))
    fig.update_layout(
        legend_title='Cluster',
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(showgrid=False, range=x_range),
        yaxis=dict(showgrid=False, range=y_range)
    )

    # CHANGE HERE: Save path for each experiment type plot
    save_path = fr"C:\Users\gangliagurdian\Desktop\Mias Folder\Poster Files\3D_Visualizations\Control\tSNE\{exp_label}"
    pio.write_image(fig, save_path + ".png", width=1000, height=1000)
    pio.write_image(fig, save_path + ".svg", width=1000, height=1000)


    fig.show()